# ML-04 - Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/github/HimanshuSharma-2856/Flyrank-ml--internship/blob/main/work/notebooks/w03_data_contract.ipynb)](https://colab.research.google.com/github/HimanshuSharma-2856/Flyrank-ml--internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook records the contract for my Structured Content Archetype Clustering lane. I develop on the mid-panel `2026-03` partition, never on the final-month `_sample`.

The warehouse is gated. Use a Hugging Face **READ** token through the `HF_TOKEN` environment variable, Colab Secret, or the secure prompt in the setup cell; never paste a token into a notebook cell.

In [2]:
%pip -q install datasets duckdb scikit-learn

Note: you may need to restart the kernel to use updated packages.


## 1. Contract in plain words

1. **What one row means:** One row in the daily fact table is one content item for one pseudonymized client on one reporting date: `report_date × client_hash_id × content_hash_id`.
2. **Tables used:** I use `fact_content_daily_performance` for daily search and analytics measures. I use `dim_content` only when a later profile needs content metadata, and keep its identifiers as context. This notebook develops from the daily fact partition.
3. **Time window:** Development uses `month=2026-03` only, covering the dates present in that partition. The final June 2026 `_sample` is sealed and is not used for label logic.
4. **What I will rank:** I will rank content-item profiles for editorial review, using a descriptive proxy rather than claiming a causal SEO outcome. The proxy in this notebook is `needs_review_proxy`, set when a content item has zero clicks in the month while receiving impressions. The click count is used to construct the proxy, not retained as an honest feature.
5. **One deliberate exclusion:** I exclude pseudonymous IDs such as `client_hash_id` and `content_hash_id` from features. They are safe for grouping and splitting, but they would let a model memorize entities instead of learning reusable structure.

**Output:** a small content-month feature frame that helps an editor compare like with like and decide which profiles deserve a human review sample first.

In [3]:
import os
import getpass

import duckdb
from datasets import load_dataset
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN or HF_TOKEN == "hf_your_read_token":
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
if not HF_TOKEN or HF_TOKEN == "hf_your_read_token" or not HF_TOKEN.startswith("hf_"):
    raise RuntimeError(
        "Set HF_TOKEN to a real Hugging Face READ token after accepting the dataset gate. "
        "Never paste it into a saved cell."
    )

# Lazy streaming access through the Hugging Face datasets library.
ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train",
    token=HF_TOKEN,
)
print("Hugging Face streaming dataset ready:", ds)

# Column- and partition-pruned SQL access through DuckDB.
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
DAILY_MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"
DAILY_NEIGHBOR = f"read_parquet(['{REL}/fact_content_daily_performance/month=2026-02/*.parquet', '{REL}/fact_content_daily_performance/month={MONTH}/*.parquet'])"

print(f"DuckDB relation ready: {REL}")
print(f"Development partition: {MONTH}")
print("Final-month _sample is excluded from development and label logic.")

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Hugging Face streaming dataset ready: IterableDataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_shards: 18
})
DuckDB relation ready: hf://datasets/FlyRank/internship-warehouse
Development partition: 2026-03
Final-month _sample is excluded from development and label logic.


## 2. Five-feature frame and field buckets

The feature frame is one row per content item for March 2026 after requiring usable GSC data. I keep five features at most. `clicks_month` is retained only to construct the teaching proxy and is excluded from the honest feature matrix:

- `impressions_month` - knowable at the decision moment because it is accumulated search demand observed during the completed March window.
- `avg_position_month` - knowable at the decision moment because it summarizes observed GSC position during the completed March window.
- `active_days_month` - knowable at the decision moment because it counts reporting days with observed impressions in the completed March window.
- `sessions_month` - knowable at the decision moment because it is aggregated only from rows where `ga4_data_available IS TRUE` in the completed March window.
- `engagement_rate_month` - knowable at the decision moment because it is averaged only from rows where `ga4_data_available IS TRUE` in the completed March window.

**Field buckets:** the five measures are features; `needs_review_proxy` is the label/proxy and never belongs in the honest feature list; `clicks_month` is label-construction support and is excluded from the honest matrix; `report_date`, `client_hash_id`, and `content_hash_id` are context for filtering, grouping, and later client-aware splits; `gsc_data_available`, `ga4_data_available`, and the final-month `_sample` are excluded from modeling because availability flags describe coverage and the sample is the sealed outcome month.

The feature frame is intentionally small and descriptive. It does not include IDs, the proxy, its source click count, or future-month information.

In [ ]:
feature_notes = {
    "impressions_month": "completed-month search impressions",
    "avg_position_month": "completed-month average GSC position",
    "active_days_month": "days with observed impressions in the completed month",
    "sessions_month": "GA4 sessions where ga4_data_available IS TRUE",
    "engagement_rate_month": "GA4 engagement rate where ga4_data_available IS TRUE",
}
print("Five planned features:")
for name, reason in feature_notes.items():
    print(f"- {name}: {reason}")
print("clicks_month is label-construction support only and is excluded from the honest matrix.")

## 3. Three verification queries on the March 2026 partition

These are the three contract checks, in order: grain, row count/date span, and availability. The availability check deliberately uses `IS TRUE`, because the warehouse flags are three-valued and `= TRUE` or `= FALSE` can mishandle nulls.

In [4]:
# Query 1: duplicate grain probe. An empty result supports the stated daily grain.
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS row_count
    FROM {DAILY_MONTH}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Query 1 - duplicate grain rows (expected: 0):")
print(grain_check.to_string(index=False))

# Query 2: scope of the development slice.
scope_check = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS first_date,
           MAX(report_date) AS last_date
    FROM {DAILY_MONTH}
""").df()
print("\nQuery 2 - March scope:")
print(scope_check.to_string(index=False))

# Query 3: usable availability. IS TRUE is intentional; NULL is not usable.
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE
        ) AS both_available_rows
    FROM {DAILY_MONTH}
""").df()
print("\nQuery 3 - availability using IS TRUE:")
print(availability_check.to_string(index=False))

assert grain_check.empty
assert int(scope_check.loc[0, "row_count"]) > 0
assert scope_check.loc[0, "first_date"].strftime("%Y-%m") == MONTH
assert scope_check.loc[0, "last_date"].strftime("%Y-%m") == MONTH
assert int(availability_check.loc[0, "both_available_rows"]) <= int(scope_check.loc[0, "row_count"])


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 - duplicate grain rows (expected: 0):
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, row_count]
Index: []

Query 2 - March scope:
 row_count first_date  last_date
   9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Query 3 - availability using IS TRUE:
 total_rows  gsc_available_rows  ga4_available_rows  both_available_rows
    9841378             3611061              413966               364347


## 4. Build the five features, then spring the leakage trap

The feature query below aggregates the same March partition into one row per content item. It creates exactly five feature columns. The deliberate trap copies the proxy label into `leaked_label_copy`; that column is allowed only in the experiment, then removed before the honest score is kept.

**Named limitation:** this is an unbalanced panel. Clients have different history depth and the March slice is not a random sample of all content. The feature frame can support a cautious review ranking, but it cannot establish that the profiles generalize to clients or months with different data coverage.

In [6]:
feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_month,
        SUM(gsc_clicks) AS clicks_month,
        AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_month,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS active_days_month,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS sessions_month,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END)
            / NULLIF(SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END), 0)
            AS engagement_rate_month
    FROM {DAILY_MONTH}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) > 0
""").df()

feature_cols = [
    "impressions_month",
    "avg_position_month",
    "active_days_month",
    "sessions_month",
    "engagement_rate_month",
]

print(f"Feature-frame rows: {len(feature_frame):,}")
print(f"Feature count: {len(feature_cols)}")
print(feature_frame[feature_cols].head().to_string(index=False))

assert len(feature_cols) == 5
assert all(column in feature_frame.columns for column in feature_cols)
assert "clicks_month" not in feature_cols
assert "client_hash_id" not in feature_cols
assert "content_hash_id" not in feature_cols

# Proxy label: a page with observed impressions but no clicks in the same completed month.
model_frame = feature_frame.dropna(subset=feature_cols + ["clicks_month"]).copy()
model_frame["needs_review_proxy"] = (
    (model_frame["impressions_month"] > 0)
    & (model_frame["clicks_month"] == 0)
).astype(int)

print(f"Rows used for the leakage check: {len(model_frame):,}")
print(f"Proxy-positive rate: {model_frame['needs_review_proxy'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame rows: 176,738
Feature count: 5
 impressions_month  avg_position_month  active_days_month  sessions_month  engagement_rate_month
              27.0            5.629630                 11             0.0                    NaN
              73.0           56.613278                 27             0.0                    NaN
             124.0           32.991931                 28             1.0                    0.0
             118.0           17.443723                 30             0.0                    NaN
             538.0            4.295309                 31             0.0                    NaN
Rows used for the leakage check: 63,601
Proxy-positive rate: 0.280


## 5. The deliberate leakage experiment

I intentionally add a copy of the proxy label to the feature matrix. Because the copied column is the answer itself, its score should jump toward perfect. I then delete it and keep the honest score from the five pre-decision features. This is a teaching trap, not a candidate model.

In [7]:
X_honest = model_frame[feature_cols]
y = model_frame["needs_review_proxy"]
X_train, X_test, y_train, y_test = train_test_split(
    X_honest, y, test_size=0.25, random_state=42, stratify=y
)

honest_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42),
)
honest_model.fit(X_train, y_train)
honest_score = roc_auc_score(y_test, honest_model.predict_proba(X_test)[:, 1])

leaked_frame = model_frame.copy()
leaked_frame["leaked_label_copy"] = leaked_frame["needs_review_proxy"]
leaked_features = feature_cols + ["leaked_label_copy"]
X_leaked = leaked_frame[leaked_features]
X_train_leaked, X_test_leaked, y_train_leaked, y_test_leaked = train_test_split(
    X_leaked, y, test_size=0.25, random_state=42, stratify=y
)
leaked_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42),
)
leaked_model.fit(X_train_leaked, y_train_leaked)
leaked_score = roc_auc_score(
    y_test_leaked, leaked_model.predict_proba(X_test_leaked)[:, 1]
)

print(f"Honest ROC AUC ({len(feature_cols)} features): {honest_score:.3f}")
print(f"Leaked ROC AUC (label copy included): {leaked_score:.3f}")
print("Leakage jump: {:.3f}".format(leaked_score - honest_score))

# Delete the trap before retaining the honest feature frame.
model_frame = model_frame.drop(columns=["needs_review_proxy"])
leaked_frame = leaked_frame.drop(columns=["leaked_label_copy", "needs_review_proxy"])
kept_feature_frame = model_frame[feature_cols].copy()

assert "leaked_label_copy" not in kept_feature_frame.columns
assert "needs_review_proxy" not in kept_feature_frame.columns
assert leaked_score > 0.99
assert leaked_score > honest_score
print("Leakage column removed; honest feature frame retained.")

Honest ROC AUC (5 features): 0.850
Leaked ROC AUC (label copy included): 1.000
Leakage jump: 0.150
Leakage column removed; honest feature frame retained.


## Self-check

Before submitting, confirm:

- [ ] The five plain-words contract answers are filled.
- [ ] Exactly three verification queries are visible, including availability filtered with `IS TRUE`.
- [ ] The feature frame contains five features and an available-when explanation for each.
- [ ] The label-derived column was shown in the trap, deleted, and excluded from the retained feature frame.
- [ ] The named unbalanced-panel limitation is understood.
- [ ] The notebook was run top to bottom with a real Hugging Face READ token, and the outputs were saved.